# <h1><b> NLP: TEXT TAGGING PRACTICAL <b><h1>

In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import re
import pandas as pd
import matplotlib.pyplot as plt

### Load data

In [2]:
pd.set_option("display.max_colwidth", None)
bbc_df = pd.read_csv(r"txt_files\bbc_news.csv")

In [3]:
bbc_df.head(3)

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?at_medium=RSS&at_campaign=KARANGA,"With much of the UK enduring another period of hot weather, some workers will face very high temperatures."
1,1,9267,'Liz Truss the Brief?' World reacts to UK political turmoil,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_medium=RSS&at_campaign=KARANGA,The UK's political chaos has been watched around the world - and commentators haven't held back.
2,2,7387,Rationing energy is nothing new for off-grid community,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlands-islands-62686916,https://www.bbc.co.uk/news/uk-scotland-highlands-islands-62686916?at_medium=RSS&at_campaign=KARANGA,Scoraig in the north west Highlands has long had to make careful use of its electricity supply.


In [4]:
bbc_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Unnamed: 0   1000 non-null   int64
 1   index        1000 non-null   int64
 2   title        1000 non-null   str  
 3   pubDate      1000 non-null   str  
 4   guid         1000 non-null   str  
 5   link         1000 non-null   str  
 6   description  1000 non-null   str  
dtypes: int64(2), str(5)
memory usage: 54.8 KB


In [5]:
titles_df = pd.DataFrame(bbc_df["title"])
titles_df.head(10)

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK political turmoil
2,Rationing energy is nothing new for off-grid community
3,The hunt for superyachts of sanctioned Russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 seconds
5,Red Bull found guilty of breaking Formula 1's budget cap
6,World Triathlon Championship Series: Flora Duffy beats Georgia Taylor-Brown to women's title
7,Terry Hall: Coventry scooter ride-out pays tribute to singer
8,Post Office and Fujitsu to face inquiry over Horizon scandal
9,'Pavement parking frightens me'


### Clean data

In [6]:
# lowercase
titles_df["lowercase"] = titles_df["title"].str.lower()
titles_df.head(3)

,title,lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK political turmoil,'liz truss the brief?' world reacts to uk political turmoil
2,Rationing energy is nothing new for off-grid community,rationing energy is nothing new for off-grid community


In [7]:
# stop word removal
en_stopwords = stopwords.words("english")
en_stopwords

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [8]:
titles_df["rm_stopwords"] = titles_df["lowercase"].apply(lambda data: " ".join([word for word in data.split() if word not in en_stopwords]))
titles_df.head(5)

,title,lowercase,rm_stopwords
0,Can I refuse to work?,can i refuse to work?,refuse work?
1,'Liz Truss the Brief?' World reacts to UK political turmoil,'liz truss the brief?' world reacts to uk political turmoil,'liz truss brief?' world reacts uk political turmoil
2,Rationing energy is nothing new for off-grid community,rationing energy is nothing new for off-grid community,rationing energy nothing new off-grid community
3,The hunt for superyachts of sanctioned Russian oligarchs,the hunt for superyachts of sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 seconds,platinum jubilee: 70 years of the queen in 70 seconds,platinum jubilee: 70 years queen 70 seconds


In [9]:
# punctation removal
titles_df["rm_punctuation"] = titles_df["rm_stopwords"].apply(lambda x: re.sub(r"[^\w\s]", "", x))
titles_df.head(3)


,title,lowercase,rm_stopwords,rm_punctuation
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work
1,'Liz Truss the Brief?' World reacts to UK political turmoil,'liz truss the brief?' world reacts to uk political turmoil,'liz truss brief?' world reacts uk political turmoil,liz truss brief world reacts uk political turmoil
2,Rationing energy is nothing new for off-grid community,rationing energy is nothing new for off-grid community,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community


In [10]:
# tokenize
titles_df["token_raw"] = titles_df.apply(lambda x: word_tokenize(x["title"]), axis=1)
titles_df["token_clean"] = titles_df.apply(lambda x: word_tokenize(x["rm_punctuation"]), axis=1)
titles_df.head(3)


,title,lowercase,rm_stopwords,rm_punctuation,token_raw,token_clean
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK political turmoil,'liz truss the brief?' world reacts to uk political turmoil,'liz truss brief?' world reacts uk political turmoil,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts, to, UK, political, turmoil]","[liz, truss, brief, world, reacts, uk, political, turmoil]"
2,Rationing energy is nothing new for off-grid community,rationing energy is nothing new for off-grid community,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[Rationing, energy, is, nothing, new, for, off-grid, community]","[rationing, energy, nothing, new, offgrid, community]"


In [11]:
# lemmatizing 
lem = WordNetLemmatizer()
titles_df["lemmatizing"] = titles_df.apply(lambda x: [lem.lemmatize(word) for word in x["token_clean"]], axis=1)
titles_df.head(3)


,title,lowercase,rm_stopwords,rm_punctuation,token_raw,token_clean,lemmatizing
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK political turmoil,'liz truss the brief?' world reacts to uk political turmoil,'liz truss brief?' world reacts uk political turmoil,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts, to, UK, political, turmoil]","[liz, truss, brief, world, reacts, uk, political, turmoil]","[liz, truss, brief, world, reacts, uk, political, turmoil]"
2,Rationing energy is nothing new for off-grid community,rationing energy is nothing new for off-grid community,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[Rationing, energy, is, nothing, new, for, off-grid, community]","[rationing, energy, nothing, new, offgrid, community]","[rationing, energy, nothing, new, offgrid, community]"


In [12]:
# create lists for just our tokens
tokens_raw_lst = sum(titles_df["token_raw"], [])
tokens_clean_lst = sum(titles_df["lemmatizing"], [])

In [13]:
tokens_raw_lst

['Can',
 'I',
 'refuse',
 'to',
 'work',
 '?',
 "'Liz",
 'Truss',
 'the',
 'Brief',
 '?',
 "'",
 'World',
 'reacts',
 'to',
 'UK',
 'political',
 'turmoil',
 'Rationing',
 'energy',
 'is',
 'nothing',
 'new',
 'for',
 'off-grid',
 'community',
 'The',
 'hunt',
 'for',
 'superyachts',
 'of',
 'sanctioned',
 'Russian',
 'oligarchs',
 'Platinum',
 'Jubilee',
 ':',
 '70',
 'years',
 'of',
 'the',
 'Queen',
 'in',
 '70',
 'seconds',
 'Red',
 'Bull',
 'found',
 'guilty',
 'of',
 'breaking',
 'Formula',
 '1',
 "'s",
 'budget',
 'cap',
 'World',
 'Triathlon',
 'Championship',
 'Series',
 ':',
 'Flora',
 'Duffy',
 'beats',
 'Georgia',
 'Taylor-Brown',
 'to',
 'women',
 "'s",
 'title',
 'Terry',
 'Hall',
 ':',
 'Coventry',
 'scooter',
 'ride-out',
 'pays',
 'tribute',
 'to',
 'singer',
 'Post',
 'Office',
 'and',
 'Fujitsu',
 'to',
 'face',
 'inquiry',
 'over',
 'Horizon',
 'scandal',
 "'Pavement",
 'parking',
 'frightens',
 'me',
 "'",
 'UK',
 'interest',
 'rates',
 ':',
 'How',
 'will',
 'the'

In [14]:
tokens_clean_lst

['refuse',
 'work',
 'liz',
 'truss',
 'brief',
 'world',
 'reacts',
 'uk',
 'political',
 'turmoil',
 'rationing',
 'energy',
 'nothing',
 'new',
 'offgrid',
 'community',
 'hunt',
 'superyachts',
 'sanctioned',
 'russian',
 'oligarch',
 'platinum',
 'jubilee',
 '70',
 'year',
 'queen',
 '70',
 'second',
 'red',
 'bull',
 'found',
 'guilty',
 'breaking',
 'formula',
 '1',
 'budget',
 'cap',
 'world',
 'triathlon',
 'championship',
 'series',
 'flora',
 'duffy',
 'beat',
 'georgia',
 'taylorbrown',
 'womens',
 'title',
 'terry',
 'hall',
 'coventry',
 'scooter',
 'rideout',
 'pay',
 'tribute',
 'singer',
 'post',
 'office',
 'fujitsu',
 'face',
 'inquiry',
 'horizon',
 'scandal',
 'pavement',
 'parking',
 'frightens',
 'me',
 'uk',
 'interest',
 'rate',
 'rise',
 'affect',
 'high',
 'could',
 'go',
 'stayed',
 'storm',
 'happens',
 'now',
 'six',
 'nation',
 'scotland',
 'best',
 'since',
 '99',
 'beat',
 'best',
 'ireland',
 'ever',
 'long',
 'liz',
 'truss',
 'survive',
 'prime',
 'm

### POS Tagging

In [15]:
nlp = spacy.load('en_core_web_sm')

In [16]:
# create a spacy doc from our raw text - better for pos tagging
# spacy_doc = nlp(' '.join(tokens_raw_lst))
spacy_doc = nlp(' '.join(tokens_clean_lst))

In [17]:
# extract the tokens and pos tags into a dataframe
pos_df = pd.DataFrame(columns=["tokens", "pos_tags"])
for token in spacy_doc:
    pos_df = pd.concat([pos_df, pd.DataFrame.from_records([{'tokens': token.text, "pos_tags": token.pos_}])], ignore_index=True)
pos_df.head(20)

,tokens,pos_tags
0,refuse,AUX
1,work,NOUN
2,liz,PROPN
3,truss,ADJ
4,brief,ADJ
5,world,NOUN
6,reacts,VERB
7,uk,PROPN
8,political,ADJ
9,turmoil,NOUN


In [18]:
# token frequency count
pos_df_count = pos_df.groupby(["tokens", "pos_tags"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False)
pos_df_count.head(15)

,tokens,pos_tags,counts
30,2022,NUM,47
1162,england,PROPN,45
870,cup,PROPN,39
3056,say,VERB,37
3707,uk,PROPN,37
3840,war,NOUN,34
2386,new,ADJ,31
3948,world,NOUN,30
3949,world,PROPN,26
3710,ukraine,PROPN,23


In [19]:
pos_frequency = pos_df_count.groupby(["pos_tags"])["tokens"].count().sort_values(ascending=False)
pos_frequency.head(15)

pos_tags
NOUN     1390
PROPN    1303
VERB      632
ADJ       403
NUM        86
ADV        61
ADP        36
AUX        33
PRON       22
DET        10
SCONJ       7
INTJ        4
PART        4
X           3
Name: tokens, dtype: int64

In [20]:
# most common nouns
nouns = pos_df_count[pos_df_count.pos_tags == "NOUN"]
nouns.head(15)

,tokens,pos_tags,counts
3840,war,NOUN,34
3948,world,NOUN,30
2136,man,NOUN,22
907,day,NOUN,21
3973,year,NOUN,20
1158,energy,NOUN,17
2847,record,NOUN,17
2657,police,NOUN,16
3870,week,NOUN,16
3935,woman,NOUN,16


In [21]:
# most common verbs
verbs = pos_df_count[pos_df_count.pos_tags == "VERB"]
verbs.head(15)

,tokens,pos_tags,counts
3056,say,VERB,37
3711,ukraine,VERB,22
1380,found,VERB,13
3461,take,VERB,13
1651,hit,VERB,13
358,beat,VERB,13
2133,make,VERB,13
1459,get,VERB,13
1473,give,VERB,11
3918,win,VERB,10


In [22]:
# most common adjectives
adjectives = pos_df_count[pos_df_count.pos_tags == "ADV"]
adjectives.head(15)

,tokens,pos_tags,counts
311,back,ADV,8
3340,still,ADV,5
2322,much,ADV,4
177,almost,ADV,4
1379,forward,ADV,4
1085,dy,ADV,3
2838,really,ADV,3
303,away,ADV,3
1197,ever,ADV,3
2432,now,ADV,2


In [23]:
from IPython.display import HTML, display
from spacy import displacy

In [24]:
def html_visiualizer(doc):
    html = displacy.render(doc, style="ent", jupyter=False)
    display(HTML(html))

### NER

In [25]:
# extract the tokens and entity tags into a dataframe
ner_df = pd.DataFrame(columns=["tokens", "labels"])
for token in spacy_doc.ents:
    ner_df = pd.concat([ner_df, pd.DataFrame.from_records([{"tokens": token.text, "labels": token.label_}])], ignore_index=True)

In [26]:
ner_df.head(10)

,tokens,labels
0,russian,NORP
1,70 year,DATE
2,70 second,TIME
3,bull,ORG
4,1,CARDINAL
5,georgia taylorbrown womens,ORG
6,terry hall,PERSON
7,six,CARDINAL
8,99,CARDINAL
9,jubilee beacon,PERSON


In [27]:
# token frequency count
ner_df_counts = ner_df.groupby(["tokens", "labels"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False)
ner_df_counts.head(5)

,tokens,labels,counts
34,2022,CARDINAL,30
451,russian,NORP,25
223,first,ORDINAL,15
35,2022,DATE,11
233,france,GPE,10


In [28]:
labels_frequency = ner_df_counts.groupby(["labels"])["tokens"].count().sort_values(ascending=False)
labels_frequency.head(15)

labels
PERSON      199
GPE          90
ORG          88
CARDINAL     72
DATE         63
NORP         37
TIME         11
LOC           7
FAC           6
ORDINAL       5
QUANTITY      3
PRODUCT       2
EVENT         1
PERCENT       1
Name: tokens, dtype: int64

In [29]:
# most common people
people = ner_df_counts[ner_df_counts.labels == "PERSON"]
people.head(15)

,tokens,labels,counts
432,putin,PERSON,7
127,boris johnson,PERSON,5
106,antonio conte,PERSON,3
102,andy murray,PERSON,3
524,tyre nichols,PERSON,2
581,zelensky,PERSON,2
249,harry,PERSON,2
254,hodgkinson,PERSON,2
252,harry meghan,PERSON,2
331,lenny henry,PERSON,2


In [30]:
# most common places
places = ner_df_counts[ner_df_counts.labels == "LOC"]
places.head(15)


,tokens,labels,counts
218,europe,LOC,2
174,crystal palace,LOC,1
91,africa,LOC,1
367,middle east,LOC,1
377,mount everest,LOC,1
401,north sea,LOC,1
564,west ham,LOC,1


In [31]:
html_visiualizer(spacy_doc)